# Width threshold at N=12 — locating the phase transition

**Purpose.** Width sweep showed w=50→42%, w=75→24%, w=100→0.86%.
Clear transition somewhere between 75 and 100. This fills in w=80, 85, 90, 95.

**Why this is a rerun.** Previous attempt got ~87% at all four widths (broken pipeline).
These should fall monotonically between 24% and 0.86%.

**Approx runtime:** ~4 h on Kaggle T4 (~55–70 min per width).

In [ ]:
import os, sys, json, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch

for p in ["/kaggle/input/kdv-core", "/kaggle/working", "."]:
    if (Path(p) / "kdv_core.py").exists():
        sys.path.insert(0, p)
        break
import kdv_core as K
print(f"device = {K.DEVICE}")

OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUT.mkdir(parents=True, exist_ok=True)
print(f"output dir = {OUT}")


In [ ]:
K.quick_sanity_check(seed=99, adam_iter=500)

### Sweep widths 80, 85, 90, 95 at N=12

In [ ]:
N = 12
SEED = 99
WIDTHS = [80, 85, 90, 95]

cfg = K.make_config(N)
K.print_config(cfg)

results = []
for width in WIDTHS:
    tag = f"N{N}_w{width}"
    ckpt_path = OUT / f"checkpoint_{tag}.pt"

    if ckpt_path.exists():
        print(f"\n[resume] {tag} done, loading.")
        ck = K.load_checkpoint(ckpt_path)
        hist = ck["history"]
        results.append(dict(width=width, params=K.PINN(width=width).n_params(),
                            L2_adam=hist["adam_l2"], L2_final=hist["lbfgs_l2"],
                            adam_time=hist["adam_time"], lbfgs_time=hist["lbfgs_time"]))
        continue

    print("\n" + "=" * 60)
    print(f"N=12, WIDTH={width}")
    print("=" * 60)
    K.set_seed(SEED)
    batch = K.build_data(cfg, seed=SEED)
    m = K.PINN(width=width).to(K.DEVICE)
    print(f"  params = {m.n_params():,}")
    hist = K.train(m, batch, adam_iter=15000, lbfgs_iter=2000)
    K.save_checkpoint(ckpt_path, m, hist, cfg,
                      extras=dict(tag=tag, width=width, N=N, seed=SEED))
    print(f"  saved {ckpt_path}")

    results.append(dict(width=width, params=m.n_params(),
                        L2_adam=hist["adam_l2"], L2_final=hist["lbfgs_l2"],
                        adam_time=hist["adam_time"], lbfgs_time=hist["lbfgs_time"]))
    pd.DataFrame(results).to_csv(OUT / "width_threshold_results.csv", index=False)

df = pd.DataFrame(results)
print("\n--- Results ---")
print(df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
df.to_csv(OUT / "width_threshold_results.csv", index=False)


### Combine with known anchors from prior runs

In [ ]:
known = pd.DataFrame([
    dict(width=50,  params=12951, L2_final=42.1502, source="scaling"),
    dict(width=75,  params=28801, L2_final=24.3948, source="width sweep"),
    dict(width=100, params=50901, L2_final=0.8554,  source="width sweep"),
])
df_new = df[["width","params","L2_final"]].copy(); df_new["source"] = "this notebook"
full = (pd.concat([known, df_new], ignore_index=True)
          .sort_values("width").reset_index(drop=True))
print(full.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
full.to_csv(OUT / "width_full_curve.csv", index=False)


### Plot

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
colors = {"scaling": "C0", "width sweep": "C1", "this notebook": "C2"}
markers = {"scaling": "o", "width sweep": "s", "this notebook": "^"}
for src in full["source"].unique():
    sub = full[full["source"] == src]
    ax.scatter(sub["width"], sub["L2_final"],
               marker=markers[src], s=100, color=colors[src],
               label=src, zorder=5, edgecolors="k")
ax.plot(full["width"], full["L2_final"], "-", color="grey", alpha=0.4, zorder=1)
for _, r in full.iterrows():
    ax.annotate(f"{r['L2_final']:.2f}%", (r["width"], r["L2_final"]),
                textcoords="offset points", xytext=(7, 6), fontsize=8)
ax.axhline(5.0, ls="--", color="red", alpha=0.6, label="5% reference")
ax.set_yscale("log")
ax.set_xlabel("hidden width"); ax.set_ylabel("final L2 error (%)")
ax.set_title(f"N={N} width sweep — locating the optimisation transition")
ax.legend(); ax.grid(alpha=0.3, which="both")
fig.tight_layout()
fig.savefig(OUT / "width_threshold_curve.png", dpi=140, bbox_inches="tight")
plt.show()


### Locate the transition

In [ ]:
full_s = full.sort_values("width").reset_index(drop=True)
print("Width step changes:")
for i in range(1, len(full_s)):
    prev, curr = full_s.iloc[i-1], full_s.iloc[i]
    drop = prev["L2_final"] - curr["L2_final"]
    print(f"  w={int(prev['width']):3d} -> w={int(curr['width']):3d}: "
          f"{prev['L2_final']:6.2f}% -> {curr['L2_final']:6.2f}%  (drop={drop:+.2f})")
diffs = full_s["L2_final"].diff(); idx = diffs.idxmin()
if idx and idx > 0:
    a, b = full_s.iloc[int(idx)-1], full_s.iloc[int(idx)]
    print(f"\nLargest drop: w={int(a['width'])} -> w={int(b['width'])}: "
          f"{a['L2_final']:.2f}% -> {b['L2_final']:.2f}%")
print("\nDone. Outputs in", OUT)
